### 1. Setup
We start by loading environment variables and initializing the Gemini client using the `google-genai` SDK.

In [1]:
# Load env variables and create client
from dotenv import load_dotenv
from google import genai

load_dotenv()

client = genai.Client()
model = "gemini-2.0-flash"

### 2. Helper Functions
These helper functions manage the conversation history and facilitate streaming interactions with the Gemini model.

In [2]:

def add_user_message(messages, message):
    if isinstance(message, list):
        messages.append({"role": "user", "parts": message})
    else:
        messages.append({"role": "user", "parts": [{"text": message}]})

def add_assistant_message(messages, response):
    messages.append(response.candidates[0].content)

def chat_stream(messages, system=None, temperature=1.0, tools=None, tool_config=None):
    from google import genai
    from google.genai import types

    config_dict = {
        "temperature": temperature,
    }
    if system:
        config_dict["system_instruction"] = system
    if tools:
        config_dict["tools"] = tools
    if tool_config:
        config_dict["tool_config"] = tool_config
        
    config = types.GenerateContentConfig(**config_dict)
    
    return client.models.generate_content_stream(
        model=model,
        contents=messages,
        config=config
    )

def text_from_message(message):
    if hasattr(message, "parts"):
        return "\n".join([part.text for part in message.parts if part.text])
    return ""


### 3. Defining Tools
Gemini can natively parse Python functions with docstrings as tool schemas. Here, we define a simple `save_article` tool.

In [3]:
from google.genai import types

def save_article(abstract: str, word_count: int, review: str):
    """"Saves a scholarly journal article

    Args:
        abstract: Abstract of the article. One short sentence max
        word_count: Word count
        review: Eight sentence review of the paper
    """
    return "Article saved!"

# Gemini natively supports parsing python functions as tools
tools = [save_article]


### 4. Executing Tool Calls
This function processes function call requests from Gemini, executes the corresponding Python logic, and returns the results as specialized `Part` objects.

In [4]:
import json

def run_tools(function_calls):
    tool_results = []
    for call in function_calls:
        try:
            if call.name == "save_article":
                result = save_article(**call.args)
            else:
                result = "Error: Unknown function"
                
            tool_results.append(
                types.Part.from_function_response(
                    name=call.name,
                    response={"result": result}
                )
            )
        except Exception as e:
            tool_results.append(
                types.Part.from_function_response(
                    name=call.name,
                    response={"error": str(e)}
                )
            )
    return tool_results

### 5. Managing the Conversation Loop
The `run_conversation` function handles the iterative process of sending messages, streaming responses, and executing tool calls until the model provides a final answer.

In [5]:
from google.genai import types

def run_conversation(messages, tools=None, tool_config=None):
    
    while True:
        response_stream = chat_stream(
            messages,
            tools=tools,
            temperature=1.0,
            tool_config=tool_config
        )
        
        full_response_parts = []
        function_calls = []
        
        print("Assistant: ", end="")
        for chunk in response_stream:
            if chunk.candidates[0].content.parts:
                for part in chunk.candidates[0].content.parts:
                    if part.text:
                        print(part.text, end="", flush=True)
                        full_response_parts.append(part)
                    elif part.function_call:
                        print(f"\n[Tool Call] {part.function_call.name}({part.function_call.args})")
                        function_calls.append(part.function_call)
                        full_response_parts.append(part)
        print("\n")
        
        # Save the assistant message
        messages.append({"role": "model", "parts": full_response_parts})
        
        if not function_calls:
            break
            
        tool_results = run_tools(function_calls)
        messages.append({"role": "user", "parts": tool_results})
        
    return messages

### 6. Example Execution
We start the conversation by asking Gemini to create and save a fake computer science article.

In [6]:
messages = []

messages.append({"role": "user", "parts": [{"text": "Create and save a fake computer science article"}]})

run_conversation(
    messages,
    tools=[save_article]
)

### 7. Multi-turn Interaction
Continuing the conversation to see how the assistant handles follow-up instructions while maintaining tool access.

In [7]:
messages.append({
    "role": "user", 
    "parts": [{"text": "Please make them up yourself. It can be about AI. Make the abstract a single sentence, word count around 5000, and give it an 8-sentence review that is generally positive but notes some performance limitations."}]
})

run_conversation(
    messages,
    tools=[save_article]
)